In [ ]:
"""
=============================================================================
PANEL MAESTRO — SCHOOL DROPOUT PREDICTION COLOMBIA
=============================================================================
Output: Data/Processed/panel_maestro.parquet + diagnostico_panel.xlsx
Primary key: SEDE_CODIGO x PERIODO_ANIO
Language: Python 3.10+
Dependencies: pip install pandas numpy openpyxl pyarrow
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import unicodedata
warnings.filterwarnings("ignore")

# =============================================================================
# 0. PATHS & CONFIG
# =============================================================================
ROOT = Path(r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data")
RAW        = ROOT / "Raw"
PROCESSED  = ROOT / "Processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

YEARS = [2018, 2019, 2020, 2021, 2022]

# C-600 file names (without year suffix)
C600_FILES = {
    "jornadas"    : "Jornadas_nivel",       # skeleton — most complete coverage
    "desplazados" : "Desplazados",
    "limitacion"  : "Limitacion_fisica",
    "tradicional" : "Ed_tradicional",
    "flexible"    : "Ed_Flexible",
    "etnia"       : "Etnia",
}

# IPM region mapping (DANE official)
REGION_MAP = {
    1:"Caribe", 2:"Oriental", 3:"Central", 4:"Bogota",
    5:"Antioquia", 6:"Valle", 7:"Pacifica", 8:"Orinoquia", 9:"San_Andres"
}
DPTO_TO_REGION = {
    "08":"Caribe","13":"Caribe","20":"Caribe","23":"Caribe","44":"Caribe","47":"Caribe","70":"Caribe",
    "15":"Oriental","25":"Oriental","54":"Oriental","68":"Oriental",
    "17":"Central","18":"Central","41":"Central","63":"Central","66":"Central","73":"Central",
    "11":"Bogota","05":"Antioquia","76":"Valle",
    "19":"Pacifica","27":"Pacifica","52":"Pacifica",
    "81":"Orinoquia","85":"Orinoquia","86":"Orinoquia","91":"Orinoquia",
    "94":"Orinoquia","95":"Orinoquia","97":"Orinoquia","99":"Orinoquia",
    "88":"San_Andres",
}

# =============================================================================
# 0b. HELPER FUNCTIONS
# =============================================================================
def check(msg, df=None, extra=None):
    print(f"\n{'='*60}")
    print(f"  CHECK: {msg}")
    if df is not None:
        print(f"  Rows: {len(df):,}  |  Cols: {df.shape[1]}  |  Nulls: {df.isnull().sum().sum():,}")
    if extra:
        print(f"  {extra}")
    print(f"{'='*60}")

def fix_sede_codigo(series):
    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(12)
    )

def normalize_str(s):
    return (
        s.str.upper().str.strip()
        .apply(lambda x: unicodedata.normalize("NFKD", str(x))
               .encode("ascii", errors="ignore").decode("ascii"))
        .str.replace(r"\s+", " ", regex=True)
    )

def find_file_flexible(folder, keyword):
    """Find first file in folder whose name contains keyword (case-insensitive)."""
    folder = Path(folder)
    if not folder.exists():
        return None
    matches = [f for f in folder.glob("*") if keyword.lower() in f.name.lower() and f.is_file()]
    return matches[0] if matches else None

def read_data_safe(path):
    """Robust reader for C-600 files (CSV and Excel). Auto-detects encoding and separator."""
    path = Path(path)
    if not path.exists():
        print(f"  [WARN] File not found: {path.name}")
        return None

    if path.suffix.lower() in [".xlsx", ".xls"]:
        try:
            df = pd.read_excel(path)
            df.columns = [
                str(c).strip()
                .replace("\ufeff", "").replace("\u200b", "")
                .upper() for c in df.columns
            ]
            return df
        except Exception as e:
            print(f"  [ERROR] Excel read failed {path.name}: {e}")
            return None

    # CSV: try encodings and separators
    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
        for sep in [",", ";"]:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep,
                                 low_memory=False, on_bad_lines="skip")
                if len(df.columns) <= 1:
                    continue
                df.columns = [
                    str(c).strip()
                    .replace("\ufeff", "").replace("\u200b", "")
                    .strip('"').strip("'").upper()
                    for c in df.columns
                ]
                print(f"  [OK] {path.name} | enc:{enc} | sep:'{sep}' | {len(df):,} rows")
                return df
            except UnicodeDecodeError:
                break
            except Exception:
                continue

    print(f"  [ERROR] Could not read: {path.name}")
    return None
# =============================================================================
# 1. LOAD & CONSOLIDATE C-600 — SKELETON (Jornadas_nivel)
# =============================================================================
print("\n>>> MODULE 1: LOADING C-600 — SKELETON (Jornadas_nivel)")

skeleton_frames = []
for yr in YEARS:
    year_dir = RAW / "C-600" / str(yr)
    if not year_dir.exists():
        year_dir = RAW / "C-600"

    # Buscar cualquier archivo que contenga "jornada" o "nivel"
    file_path = find_file_flexible(year_dir, "jornada")
    if file_path is None:
        file_path = find_file_flexible(year_dir, "nivel")

    if file_path is None:
        print(f"  [WARN] No se encontró archivo esqueleto en {year_dir}")
        print(f"         Archivos detectados en la carpeta: {[f.name for f in year_dir.glob('*') if f.is_file()]}")
        continue

    print(f"  Cargando {yr}: {file_path.name}")
    df = read_data_safe(file_path)
    if df is None or df.empty:
        continue

    # Identificar columna SEDE_CODIGO dinámicamente
    col_sede = next((c for c in df.columns if "SEDE" in c and "COD" in c), None)
    if not col_sede:
        print(f"  [WARN] No se encontró columna SEDE_CODIGO en {file_path.name}. Columnas: {list(df.columns)}")
        continue

    df["SEDE_CODIGO"] = fix_sede_codigo(df[col_sede])

    print(
    f"  [DEBUG] Sedes válidas en {yr}: "
    f"{df['SEDE_CODIGO'].notna().sum():,}"
    )

    print(
    f"  [DEBUG] Sedes únicas en {yr}: "
    f"{df['SEDE_CODIGO'].nunique():,}"
    )

    df["PERIODO_ANIO"] = yr

    df["COD_MPIO_DANE"] = (
    df["SEDE_CODIGO"]
    .astype("string")
    .str[:5]
    )

    df["COD_DPTO_DANE"] = (
    df["SEDE_CODIGO"]
    .astype("string")
    .str[:2]
    )

    # Calcular MATRICULA_TOTAL (Suma total directa o Hombres + Mujeres)
    col_cant = next((c for c in df.columns if "SEDEALUM" in c or "CANTIDAD_TOTAL" in c), None)
    cols_hm = [c for c in df.columns if "CANTIDAD_HOMBRE" in c or "CANTIDAD_MUJER" in c or "HOMBRE" in c or "MUJER" in c]

    if col_cant:
        df["MATRICULA_TOTAL"] = pd.to_numeric(df[col_cant], errors="coerce").fillna(0)
    elif cols_hm:
        for c in cols_hm:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        df["MATRICULA_TOTAL"] = df[cols_hm].sum(axis=1)
    else:
        df["MATRICULA_TOTAL"] = 0

    # Agrupar por sede-año
    agg = (
        df.groupby(["SEDE_CODIGO", "PERIODO_ANIO", "COD_MPIO_DANE", "COD_DPTO_DANE"])
        ["MATRICULA_TOTAL"].sum()
        .reset_index()
    )
    skeleton_frames.append(agg)

if not skeleton_frames:
    raise FileNotFoundError(
        f"ERROR CRÍTICO: No se logró consolidar ninguna sede.\n"
        f"Verifica la ruta: {RAW / 'C-600'}"
    )

panel = pd.concat(skeleton_frames, ignore_index=True)

check("Skeleton built (Jornadas_nivel)", panel,
      f"Unique sedes: {panel['SEDE_CODIGO'].nunique():,} | Years: {sorted(panel['PERIODO_ANIO'].unique())}")

# =============================================================================
# 2. LOAD & JOIN REMAINING C-600 SOURCES
# =============================================================================
print("\n>>> MODULE 2: LOADING & JOINING REMAINING C-600 SOURCES")

C600_CONTEO_COLS = {
    "desplazados": ("JORNDES_CANTIDAD_HOMBRE", "JORNDES_CANTIDAD_MUJER"),
    "limitacion" : ("JORNLIM_CANTIDAD_HOMBRE", "JORNLIM_CANTIDAD_MUJER"),
    "tradicional": ("JORNTRA_CANTIDAD_HOMBRE", "JORNTRA_CANTIDAD_MUJER"),
    "flexible"   : ("JORNMOD_CANTIDAD_HOMBRE", "JORNMOD_CANTIDAD_MUJER"),
    "etnia"      : ("JORNETN_CANTIDAD_HOMBRE", "JORNETN_CANTIDAD_MUJER"),
}

for key, (col_h, col_m) in C600_CONTEO_COLS.items():
    frames = []
    for yr in YEARS:
        fname = C600_FILES[key]
        path = RAW / "C-600" / str(yr) / f"{fname}_{yr}.csv"
        df = read_data_safe(path)
        if df is None:
            continue
        df["SEDE_CODIGO"] = fix_sede_codigo(df["SEDE_CODIGO"])
        df["PERIODO_ANIO"] = yr

        # Numeric conversion
        for c in [col_h, col_m]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

        # Aggregate to sede-year
        cols_present = [c for c in [col_h, col_m] if c in df.columns]
        if not cols_present:
            print(f"  [WARN] {key} {yr}: conteo columns not found")
            continue

        agg = (
            df.groupby(["SEDE_CODIGO", "PERIODO_ANIO"])[cols_present]
            .sum()
            .reset_index()
            .rename(columns={
                col_h: f"{key.upper()}_H",
                col_m: f"{key.upper()}_M",
            })
        )
        agg[f"{key.upper()}_TOTAL"] = agg[[f"{key.upper()}_H", f"{key.upper()}_M"]].sum(axis=1)
        frames.append(agg)

    if not frames:
        print(f"  [WARN] {key}: no data loaded for any year")
        continue

    df_source = pd.concat(frames, ignore_index=True)
    panel = panel.merge(df_source, on=["SEDE_CODIGO", "PERIODO_ANIO"], how="left")
    match_pct = panel[f"{key.upper()}_TOTAL"].notna().mean() * 100
    print(f"  OK  {key:<14}: {match_pct:.1f}% of sede-years have data")

check("C-600 lateral joins complete", panel,
      f"Columns so far: {list(panel.columns)}")

# =============================================================================
# 3. COMPUTE COMPOSITION FEATURES
# =============================================================================
print("\n>>> MODULE 3: COMPUTING COMPOSITION & VULNERABILITY FEATURES")

for key in ["DESPLAZADOS", "LIMITACION", "TRADICIONAL", "FLEXIBLE", "ETNIA"]:
    col_total = f"{key}_TOTAL"
    if col_total in panel.columns:
        panel[f"PROP_{key}"] = np.where(
            panel["MATRICULA_TOTAL"] > 0,
            (panel[col_total] / panel["MATRICULA_TOTAL"]).clip(0, 1),
            np.nan
        )

# Gender index (femininity ratio)
panel["IDX_FEMINIDAD"] = np.where(
    panel["MATRICULA_TOTAL"] > 0,
    (panel.get("DESPLAZADOS_M", 0) + panel.get("ETNIA_M", 0) +
     panel.get("TRADICIONAL_M", 0) + panel.get("FLEXIBLE_M", 0)) / panel["MATRICULA_TOTAL"],
    np.nan
)

# Composite vulnerability index (literature-based weights)
# Desplazados x2 (highest dropout risk), Limitacion x1.5, Etnia x1, Flexible x0.5
weights = {"PROP_DESPLAZADOS": 2.0, "PROP_LIMITACION": 1.5,
           "PROP_ETNIA": 1.0, "PROP_FLEXIBLE": 0.5}
panel["INDICE_VULNERABILIDAD"] = sum(
    panel[c].fillna(0) * w for c, w in weights.items() if c in panel.columns
)

# Pandemic flag
panel["FLAG_PANDEMIA"] = (panel["PERIODO_ANIO"] == 2020).astype(int)

check("Composition features computed", panel,
      f"Mean vulnerability index: {panel['INDICE_VULNERABILIDAD'].mean():.4f}")

# =============================================================================
# 4. SIMAT — MUNICIPAL DROPOUT & REPETITION RATES
# =============================================================================
print("\n>>> MODULE 4: LOADING SIMAT (municipal rates)")

def load_simat(filename, conteo_col):
    """Load SIMAT Excel, filter to MUNICIPIO + Oficial, return clean DataFrame."""
    path = RAW / "SIMAT" / filename
    if not path.exists():
        print(f"  [ERROR] Not found: {filename}")
        return pd.DataFrame()
    df = pd.read_excel(path, sheet_name=0, skiprows=5)
    df.columns = df.columns.str.strip().str.upper().str.replace(" ", "_")
    df = df.rename(columns={"AÑO": "ANIO", "A_O": "ANIO"})

    # Find year column flexibly
    yr_col = next((c for c in df.columns if "A" in c and "O" in c and len(c) <= 4), None)
    if yr_col and yr_col != "ANIO":
        df = df.rename(columns={yr_col: "ANIO"})

    df["ANIO"] = pd.to_numeric(df.get("ANIO", np.nan), errors="coerce")
    df = df[
        (df.get("TERRITORIO", "") == "MUNICIPIO") &
        (df.get("SECTOR", "").str.lower().str.contains("oficial", na=False)) &
        (df["ANIO"].isin(YEARS))
    ].copy()

    df["TASA"] = pd.to_numeric(df.get("TASA", np.nan), errors="coerce")
    df["TOTAL"] = pd.to_numeric(df.get("TOTAL", np.nan), errors="coerce")
    df[conteo_col] = pd.to_numeric(df.get(conteo_col, np.nan), errors="coerce")
    df["MUNICIPIO_STD"] = normalize_str(df["MUNICIPIO"].fillna(""))
    df["DEPARTAMENTO_STD"] = normalize_str(df["DEPARTAMENTO"].fillna(""))
    return df[["ANIO", "MUNICIPIO_STD", "DEPARTAMENTO_STD", conteo_col, "TOTAL", "TASA"]]

df_desercion  = load_simat("Tasa_Desercion_intra_Departamentos.xlsx",  "DESERTORES")
df_repitencia = load_simat("Tasa_repitencia_intra_Departamentos.xlsx", "REPITENTES")

check("SIMAT loaded", df_desercion,
      f"Desercion rows: {len(df_desercion):,} | Repitencia rows: {len(df_repitencia):,}")

# Load DIVIPOLA to map municipality name -> COD_MPIO_DANE
# Load DIVIPOLA to map municipality name -> COD_MPIO_DANE
print("  Loading DIVIPOLA...")
divipola_path = RAW / "SIMAT" / "DIVIPOLA.csv"
div = pd.read_csv(divipola_path, encoding="utf-8-sig", low_memory=False)

# Normalizar encabezados (remueve tildes: CÓDIGO -> CODIGO)
div.columns = [
    unicodedata.normalize("NFKD", str(c))
    .encode("ascii", errors="ignore")
    .decode("utf-8")
    .upper()
    .strip()
    for c in div.columns
]

# Detectar columnas clave de forma segura y flexible
cod_col  = next((c for c in div.columns if "COD" in c and ("MUN" in c or "MPO" in c)), None)
nom_col  = next((c for c in div.columns if ("MUN" in c or "MPO" in c) and "COD" not in c), None)
dpto_col = next((c for c in div.columns if ("DEP" in c or "DPTO" in c) and "COD" not in c), None)

div["COD_MPIO_DANE"] = div[cod_col].astype(str).str.zfill(5)
div["MUNICIPIO_STD"]    = normalize_str(div[nom_col].fillna(""))
div["DEPARTAMENTO_STD"] = normalize_str(div[dpto_col].fillna(""))
divipola_clean = div[["COD_MPIO_DANE", "MUNICIPIO_STD", "DEPARTAMENTO_STD"]].drop_duplicates()

check("DIVIPOLA loaded", divipola_clean,
      f"Unique municipalities: {divipola_clean['COD_MPIO_DANE'].nunique():,}")

# Aggregate SIMAT to municipality-year (weighted average by TOTAL)
def agg_simat_to_mpio(df_simat, tasa_col_out, conteo_col):
    """Weighted average rate by municipality-year across education levels."""
    def wavg(g):
        mask = g["TASA"].notna() & g["TOTAL"].notna() & (g["TOTAL"] > 0)
        if mask.sum() == 0:
            return pd.Series({tasa_col_out: np.nan, f"{conteo_col}_SUM": np.nan, "TOTAL_SUM": np.nan})
        return pd.Series({
            tasa_col_out      : np.average(g.loc[mask, "TASA"], weights=g.loc[mask, "TOTAL"]),
            f"{conteo_col}_SUM": g[conteo_col].sum(),
            "TOTAL_SUM"       : g["TOTAL"].sum(),
        })
    return (
        df_simat.groupby(["ANIO", "MUNICIPIO_STD", "DEPARTAMENTO_STD"])
        .apply(wavg).reset_index()
    )

simat_desercion  = agg_simat_to_mpio(df_desercion,  "TASA_DESERCION_MPIO",  "DESERTORES")
simat_repitencia = agg_simat_to_mpio(df_repitencia, "TASA_REPITENCIA_MPIO", "REPITENTES")

# Join DIVIPOLA to get COD_MPIO_DANE
simat_desercion  = simat_desercion.merge(divipola_clean, on=["MUNICIPIO_STD","DEPARTAMENTO_STD"], how="left")
simat_repitencia = simat_repitencia.merge(divipola_clean, on=["MUNICIPIO_STD","DEPARTAMENTO_STD"], how="left")

no_match_d = simat_desercion["COD_MPIO_DANE"].isna().sum()
no_match_r = simat_repitencia["COD_MPIO_DANE"].isna().sum()
check("SIMAT aggregated & DIVIPOLA matched",
      extra=f"Desercion no-match: {no_match_d} | Repitencia no-match: {no_match_r}")

# Impute municipal rate to panel (homoscedasticity assumption)
panel = panel.merge(
    simat_desercion[["ANIO", "COD_MPIO_DANE", "TASA_DESERCION_MPIO"]],
    left_on=["PERIODO_ANIO", "COD_MPIO_DANE"],
    right_on=["ANIO", "COD_MPIO_DANE"], how="left"
).drop(columns=["ANIO"], errors="ignore")

panel = panel.merge(
    simat_repitencia[["ANIO", "COD_MPIO_DANE", "TASA_REPITENCIA_MPIO"]],
    left_on=["PERIODO_ANIO", "COD_MPIO_DANE"],
    right_on=["ANIO", "COD_MPIO_DANE"], how="left"
).drop(columns=["ANIO"], errors="ignore")

cov_d = panel["TASA_DESERCION_MPIO"].notna().mean() * 100
cov_r = panel["TASA_REPITENCIA_MPIO"].notna().mean() * 100
check("SIMAT imputed to panel",
      extra=f"Desercion coverage: {cov_d:.1f}% | Repitencia coverage: {cov_r:.1f}%")

# =============================================================================
# 5. TEMPORAL FEATURES & LAGS
# =============================================================================
print("\n>>> MODULE 5: COMPUTING TEMPORAL FEATURES & LAGS")

panel = panel.sort_values(["SEDE_CODIGO", "PERIODO_ANIO"])

panel["MATRICULA_DELTA"]       = panel.groupby("SEDE_CODIGO")["MATRICULA_TOTAL"].diff()
panel["MATRICULA_PCT_CAMBIO"]  = (
    panel["MATRICULA_DELTA"] /
    panel.groupby("SEDE_CODIGO")["MATRICULA_TOTAL"].shift(1)
).clip(-1, 5)

panel["DESERCION_LAG1"]   = panel.groupby("SEDE_CODIGO")["TASA_DESERCION_MPIO"].shift(1)
panel["REPITENCIA_LAG1"]  = panel.groupby("SEDE_CODIGO")["TASA_REPITENCIA_MPIO"].shift(1)
panel["DESERCION_MA2"]    = (
    panel.groupby("SEDE_CODIGO")["TASA_DESERCION_MPIO"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)
panel["FLAG_DECLIVE_MATRICULA"] = (
    (panel["MATRICULA_DELTA"] < 0) &
    (panel.groupby("SEDE_CODIGO")["MATRICULA_DELTA"].shift(1) < 0)
).astype(int)

check("Temporal features computed", panel,
      f"Lag1 desercion coverage: {panel['DESERCION_LAG1'].notna().sum():,} rows")

# =============================================================================
# 6. IPM — REGIONAL SOCIOECONOMIC CONTEXT
# =============================================================================
print("\n>>> MODULE 6: LOADING IPM (regional context)")

ipm_frames = []
for yr in YEARS:
    path = RAW / "IPM" / f"IPM_Hogares_{yr}.csv"
    if not path.exists():
        print(f"  [WARN] IPM {yr} not found")
        continue
    df = pd.read_csv(path, sep=";", encoding="latin1", low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
    df["anio"] = yr
    ipm_frames.append(df)

df_ipm = pd.concat(ipm_frames, ignore_index=True)
df_ipm["region"] = pd.to_numeric(df_ipm.get("region", np.nan), errors="coerce")
df_ipm["region_nombre"] = df_ipm["region"].map(REGION_MAP)

privaciones = ["inasistencia_escolar", "rezago_escolar", "trabajo_infantil",
               "hacinamiento", "empleo_formal", "alfabetismo", "logro_educativo",
               "aseguramiento_salud", "barreras_acceso_salud", "paredes", "pisos",
               "alcantarillado", "acueducto", "desempleo_larga_duracion", "atencion_integral", "ipm"]

cols_ipm = [c for c in privaciones if c in df_ipm.columns]
df_ipm["fex_c"] = pd.to_numeric(df_ipm.get("fex_c", 1), errors="coerce").fillna(1)

# Weighted average by year and region
ipm_agg = (
    df_ipm.groupby(["anio", "region_nombre"])
    .apply(lambda g: pd.Series({
        f"IPM_{c.upper()}": np.average(
            pd.to_numeric(g[c], errors="coerce").fillna(0),
            weights=g["fex_c"]
        ) for c in cols_ipm
    }))
    .reset_index()
    .rename(columns={"anio": "PERIODO_ANIO", "region_nombre": "REGION_DANE"})
)

panel["REGION_DANE"] = panel["COD_DPTO_DANE"].map(DPTO_TO_REGION)
panel = panel.merge(ipm_agg, on=["PERIODO_ANIO", "REGION_DANE"], how="left")

check("IPM merged", panel,
      f"IPM coverage: {panel['IPM_IPM'].notna().mean()*100:.1f}%")

# =============================================================================
# 7. ENRICHMENT — PDET, ZOMAC, ICFES, TERRIDATA
# =============================================================================
print("\n>>> MODULE 7: LOADING ENRICHMENT SOURCES")

# --- PDET ---
pdet_path = RAW / "Enrichment" / "PDET_municipios.xlsx"
if pdet_path.exists():
    pdet = pd.read_excel(pdet_path)
    pdet.columns = pdet.columns.str.strip().str.upper()
    cod_col = next((c for c in pdet.columns if "DANE" in c or "COD" in c), None)
    if cod_col:
        pdet["COD_MPIO_DANE"] = pdet[cod_col].astype(str).str.zfill(5)
        pdet["FLAG_PDET"] = 1
        panel = panel.merge(pdet[["COD_MPIO_DANE", "FLAG_PDET"]].drop_duplicates(),
                            on="COD_MPIO_DANE", how="left")
        panel["FLAG_PDET"] = panel["FLAG_PDET"].fillna(0).astype(int)
        print(f"  OK  PDET: {panel['FLAG_PDET'].sum():,} sede-years in PDET municipalities")
else:
    print("  [WARN] PDET file not found")

# --- ZOMAC ---
zomac_path = RAW / "Enrichment" / "ZOMAC_municipios.xlsx"
if zomac_path.exists():
    zomac = pd.read_excel(zomac_path)
    zomac.columns = zomac.columns.str.strip().str.upper()
    cod_col = next((c for c in zomac.columns if "DANE" in c or "COD" in c), None)
    if cod_col:
        zomac["COD_MPIO_DANE"] = zomac[cod_col].astype(str).str.zfill(5)
        zomac["FLAG_ZOMAC"] = 1
        panel = panel.merge(zomac[["COD_MPIO_DANE", "FLAG_ZOMAC"]].drop_duplicates(),
                            on="COD_MPIO_DANE", how="left")
        panel["FLAG_ZOMAC"] = panel["FLAG_ZOMAC"].fillna(0).astype(int)
        print(f"  OK  ZOMAC: {panel['FLAG_ZOMAC'].sum():,} sede-years in ZOMAC municipalities")
else:
    print("  [WARN] ZOMAC file not found")

check("PDET & ZOMAC joined",
      extra=f"PDET sedes: {panel.get('FLAG_PDET', pd.Series([0])).sum():,} | ZOMAC sedes: {panel.get('FLAG_ZOMAC', pd.Series([0])).sum():,}")

# --- ICFES ---
icfes_path = RAW / "Enrichment" / "Icfes_Resumen.csv"
if icfes_path.exists():
    icfes = pd.read_csv(icfes_path, encoding="latin1", low_memory=False)
    icfes.columns = icfes.columns.str.strip().str.upper()

    # Fix SEDE_CODIGO (same scientific notation issue)
    icfes["SEDE_CODIGO"] = fix_sede_codigo(icfes["COLE_COD_DANE_SEDE"])
    icfes["PERIODO_ANIO"] = pd.to_numeric(icfes["ANIO"], errors="coerce")

    icfes_cols = {
        "CANT_ESTUDIANTES"       : "ICFES_CANT_ESTUDIANTES",
        "PROM_PUNT_GLOBAL"       : "ICFES_PROM_PUNT_GLOBAL",
        "PCT_DESPLAZACOLEGIO"    : "ICFES_PCT_DESPLAZACOLEGIO",
        "PCT_HORASTRABNOREMU"    : "ICFES_PCT_HORASTRABNOREMU",
        "PCT_FAMI_TIENEINTERNET" : "ICFES_PCT_INTERNET",
    }
    cols_present = {k: v for k, v in icfes_cols.items() if k in icfes.columns}
    icfes_clean = (
        icfes[["SEDE_CODIGO", "PERIODO_ANIO"] + list(cols_present.keys())]
        .rename(columns=cols_present)
        .drop_duplicates(subset=["SEDE_CODIGO", "PERIODO_ANIO"])
    )
    panel = panel.merge(icfes_clean, on=["SEDE_CODIGO", "PERIODO_ANIO"], how="left")
    cov = panel["ICFES_PROM_PUNT_GLOBAL"].notna().mean() * 100
    print(f"  OK  ICFES: {cov:.1f}% sede-year coverage")
else:
    print("  [WARN] Icfes_Resumen.csv not found")

check("ICFES joined", panel)

# --- TERRIDATA ---
terridata_path = RAW / "Enrichment" / "Terridata_completo.csv"
if terridata_path.exists():
    print("  Loading Terridata (large file, may take a moment)...")
    terra = pd.read_csv(terridata_path, encoding="latin1", low_memory=False)
    terra.columns = terra.columns.str.strip().str.upper()

    # Normalize key columns
    terra["COD_MPIO_DANE"] = terra["CODIGO_ENTIDAD"].astype(str).str.zfill(5)
    terra["PERIODO_ANIO"]  = pd.to_numeric(terra["ANIO"], errors="coerce")
    terra["DATO"]          = pd.to_numeric(terra["DATO"], errors="coerce")
    terra = terra[terra["PERIODO_ANIO"].isin(YEARS)]

    print(f"  Terridata indicators available: {terra['INDICADOR'].nunique():,}")
    print(f"  Sample indicators:\n{terra['INDICADOR'].value_counts().head(10).to_string()}")

    # Pivot each indicator as a column (prefix TERRA_)
    terra_pivot = (
        terra.pivot_table(
            index=["COD_MPIO_DANE", "PERIODO_ANIO"],
            columns="INDICADOR",
            values="DATO",
            aggfunc="mean"
        )
        .reset_index()
    )
    # Clean column names
    terra_pivot.columns = (
        ["COD_MPIO_DANE", "PERIODO_ANIO"] +
        ["TERRA_" + str(c).upper().replace(" ", "_")[:50]
         for c in terra_pivot.columns[2:]]
    )
    panel = panel.merge(terra_pivot, on=["COD_MPIO_DANE", "PERIODO_ANIO"], how="left")
    terra_cols = [c for c in panel.columns if c.startswith("TERRA_")]
    print(f"  OK  Terridata: {len(terra_cols)} indicators added")
else:
    print("  [WARN] Terridata_completo.csv not found")

check("All enrichment sources joined", panel,
      f"Total columns: {panel.shape[1]}")

# =============================================================================
# 8. FINAL CLEANUP & COLUMN ORDERING
# =============================================================================
print("\n>>> MODULE 8: FINAL CLEANUP")

# Drop purely technical/redundant ID columns
drop_patterns = ["_ID", "_CODIGO", "PERIODO_ID"]
cols_to_drop  = [c for c in panel.columns
                 if any(p in c for p in drop_patterns) and c != "SEDE_CODIGO"]
panel = panel.drop(columns=cols_to_drop, errors="ignore")

# Logical column order
cols_key        = ["SEDE_CODIGO", "PERIODO_ANIO", "COD_MPIO_DANE", "COD_DPTO_DANE", "REGION_DANE"]
cols_target     = ["TASA_DESERCION_MPIO", "TASA_REPITENCIA_MPIO"]
cols_lags       = ["DESERCION_LAG1", "REPITENCIA_LAG1", "DESERCION_MA2"]
cols_matricula  = [c for c in panel.columns if "MATRICULA" in c or "FLAG" in c]
cols_c600       = [c for c in panel.columns if any(k in c for k in
                   ["DESPLAZADOS","LIMITACION","TRADICIONAL","FLEXIBLE","ETNIA","PROP_","INDICE","IDX_"])]
cols_icfes      = [c for c in panel.columns if c.startswith("ICFES_")]
cols_ipm        = [c for c in panel.columns if c.startswith("IPM_")]
cols_enrichment = [c for c in panel.columns if c.startswith("TERRA_") or c in ["FLAG_PDET","FLAG_ZOMAC"]]
cols_rest       = [c for c in panel.columns
                   if c not in cols_key+cols_target+cols_lags+cols_matricula
                   +cols_c600+cols_icfes+cols_ipm+cols_enrichment]


# Remove duplicated column names from the final ordering
# while preserving the intended order


final_order = (
    cols_key
    + cols_target
    + cols_lags
    + cols_matricula
    + cols_c600
    + cols_icfes
    + cols_ipm
    + cols_enrichment
    + cols_rest
)

# Remove repeated column names while preserving first occurrence
final_order = list(dict.fromkeys(final_order))

# Keep only columns that actually exist in panel
final_order = [
    c for c in final_order
    if c in panel.columns
]

panel = panel.loc[:, final_order]

panel = (
    panel
    .sort_values(["SEDE_CODIGO", "PERIODO_ANIO"])
    .reset_index(drop=True)
)


# VALIDATE COLUMN UNIQUENESS BEFORE EXPORT


duplicated_cols = panel.columns[
    panel.columns.duplicated(keep=False)
].tolist()

if duplicated_cols:

    print(
        "\n[ERROR] Duplicate columns still present:"
    )
    print(duplicated_cols)

    raise ValueError(
        "El panel todavía contiene columnas duplicadas."
    )

print(
    f"\n[OK] Column names are unique: "
    f"{panel.shape[1]} columns"
)

check("Final panel ready", panel,
      f"Unique sedes: {panel['SEDE_CODIGO'].nunique():,} | Rows: {len(panel):,} | Cols: {panel.shape[1]}")

# =============================================================================
# 9. QUALITY DIAGNOSTICS
# =============================================================================
print("\n>>> MODULE 9: GENERATING QUALITY REPORT")

diag = pd.DataFrame({
    "Variable"   : panel.columns,
    "Tipo"       : panel.dtypes.astype(str).values,
    "N_nulos"    : panel.isnull().sum().values,
    "Pct_nulos"  : (panel.isnull().mean() * 100).round(2).values,
    "Media"      : [panel[c].mean()  if pd.api.types.is_numeric_dtype(panel[c]) else None for c in panel.columns],
    "Min"        : [panel[c].min()   if pd.api.types.is_numeric_dtype(panel[c]) else None for c in panel.columns],
    "Max"        : [panel[c].max()   if pd.api.types.is_numeric_dtype(panel[c]) else None for c in panel.columns],
}).sort_values("Pct_nulos", ascending=False)

# Target variable correlations
target = "TASA_DESERCION_MPIO"
num_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != target and panel[c].notna().values.sum() > 500]
corr = (
    panel[num_cols + [target]].corr()[target]
    .drop(target, errors="ignore")
    .sort_values(key=abs, ascending=False)
    .reset_index()
    .rename(columns={"index": "Variable", target: "Correlacion_Pearson"})
)
corr["Correlacion_Pearson"] = corr["Correlacion_Pearson"].round(4)

print(f"\n  Top 15 variables correlated with TASA_DESERCION_MPIO:")
print(corr.head(15).to_string(index=False))

lag_corr = panel[[target, "DESERCION_LAG1"]].dropna().corr().loc[target, "DESERCION_LAG1"]
print(f"\n  Autocorrelation lag-1 (R): {lag_corr:.4f}  |  R²: {lag_corr**2:.4f}")

# =============================================================================
# 10. EXPORT
# =============================================================================
print("\n>>> MODULE 10: EXPORTING RESULTS")

print("\n>>> MODULE 10: EXPORTING RESULTS")

# Final validation
duplicate_columns = panel.columns[
    panel.columns.duplicated(keep=False)
].tolist()

print(f"  Rows: {len(panel):,}")
print(f"  Columns: {len(panel.columns):,}")
print(f"  Duplicate columns: {len(duplicate_columns)}")

if duplicate_columns:
    print("  [ERROR] Duplicate columns:")
    print(sorted(set(duplicate_columns)))

    raise ValueError(
        "No se puede exportar: existen nombres de columnas duplicados."
    )

print("  [OK] Column names are unique.")

panel.to_parquet(
    PROCESSED / "panel_maestro.parquet",
    index=False
)

panel.to_csv(
    PROCESSED / "panel_maestro.csv",
    index=False,
    encoding="utf-8-sig"
)

panel.to_parquet(PROCESSED / "panel_maestro.parquet", index=False)
panel.to_csv(PROCESSED / "panel_maestro.csv", index=False, encoding="utf-8-sig")

with pd.ExcelWriter(PROCESSED / "diagnostico_panel.xlsx", engine="openpyxl") as writer:
    diag.to_excel(writer, sheet_name="Completitud", index=False)
    corr.to_excel(writer, sheet_name="Correlaciones_target", index=False)
    panel.describe().T.reset_index().to_excel(writer, sheet_name="Descriptivos", index=False)
    # Municipalities with no SIMAT match (for debugging)
    sin_match = panel[panel["TASA_DESERCION_MPIO"].isna()][
        ["SEDE_CODIGO","PERIODO_ANIO","COD_MPIO_DANE"]
    ].drop_duplicates().head(300)
    sin_match.to_excel(writer, sheet_name="Sin_match_SIMAT", index=False)

print(f"\n  panel_maestro.parquet  -> {PROCESSED}")
print(f"  panel_maestro.csv      -> {PROCESSED}")
print(f"  diagnostico_panel.xlsx -> {PROCESSED}")

check("PIPELINE COMPLETE — all outputs exported", panel,
      f"Final shape: {panel.shape[0]:,} rows x {panel.shape[1]} cols")


>>> MODULE 1: LOADING C-600 — SKELETON (Jornadas_nivel)
  Cargando 2018: Jornadas_nivel_2018.csv
  [OK] Jornadas_nivel_2018.csv | enc:cp1252 | sep:',' | 127,310 rows
  [DEBUG] Sedes válidas en 2018: 127,310
  [DEBUG] Sedes únicas en 2018: 53,202
  Cargando 2019: Jornadas_nivel_2019.csv
  [OK] Jornadas_nivel_2019.csv | enc:utf-8-sig | sep:';' | 130,645 rows
  [DEBUG] Sedes válidas en 2019: 130,645
  [DEBUG] Sedes únicas en 2019: 53,527
  Cargando 2020: Jornadas_nivel_2020.csv
  [OK] Jornadas_nivel_2020.csv | enc:utf-8-sig | sep:';' | 129,297 rows
  [DEBUG] Sedes válidas en 2020: 129,297
  [DEBUG] Sedes únicas en 2020: 53,484
  Cargando 2021: Jornadas_nivel_2021.CSV
  [OK] Jornadas_nivel_2021.CSV | enc:cp1252 | sep:',' | 128,432 rows
  [DEBUG] Sedes válidas en 2021: 128,432
  [DEBUG] Sedes únicas en 2021: 53,066
  Cargando 2022: Jornadas_nivel_2022.CSV
  [OK] Jornadas_nivel_2022.CSV | enc:cp1252 | sep:',' | 129,765 rows
  [DEBUG] Sedes válidas en 2022: 129,765
  [DEBUG] Sedes únicas en 